# ⚽ T-DEED: Ball Action Spotting (Pass & Shot Detection)
### End-to-End Training & Fine-Tuning Pipeline on Kaggle (SoccerNet SN-BAS-2024)

**Objective**: Replace error-prone kinematic ball-tracking heuristics with a temporal deep-learning model (**T-DEED**) to spot football events (**Passes, Shots, Goals, Crosses, Headers, Tackles**) directly from broadcast video frames.

**Why T-DEED?**:
- **Winner of SoccerNet Ball Action Spotting 2024** & baseline for 2025.
- **Robust to Ball Occlusions**: Recognizes player body language, leg swing, and collective movement rather than relying on noisy 2D ball bounding boxes.
- **12 Football Event Classes**: `Pass`, `Drive`, `Header`, `High Pass`, `Out`, `Cross`, `Throw In`, `Shot`, `Ball-Player Block`, `Player Successful Tackle`, `Free Kick`, `Goal`.
- **Lightweight Architecture**: RegNetY-200MF (`rny002_gsf`) backbone + Temporal SGP Mixer (~49 MB checkpoint, runs efficiently on Kaggle T4 GPU).

--- 
## Step 1: Install Dependencies (Run BEFORE any Python imports)
Installing all upgraded packages first prevents in-memory kernel conflicts with pre-installed Kaggle packages.

In [ ]:
# Step 1: Upgrade huggingface_hub and SoccerNet before importing anything
!pip install -q -U "huggingface_hub>=0.23.0" SoccerNet
!pip install -q einops moviepy av scikit-learn wandb
!nvidia-smi

--- 
## Step 2: Hugging Face Authentication (No Password Required)
SoccerNet now authenticates via Hugging Face access tokens (no deprecated FTP password needed).
Add your token to Kaggle **Add-ons -> Secrets** under the key `HF_TOKEN`, or enter it directly when prompted.

In [ ]:
import os

# Check for Kaggle Secret first
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    print("✅ Found HF_TOKEN in Kaggle Secrets.")
except Exception:
    hf_token = os.environ.get("HF_TOKEN", "")

if not hf_token:
    import getpass
    hf_token = getpass.getpass("Enter your Hugging Face Access Token: ")

os.environ["HF_TOKEN"] = hf_token
# Authenticate CLI and environment
!huggingface-cli login --token $hf_token
print("✅ Successfully authenticated with Hugging Face.")

--- 
## Step 3: Clone T-DEED Repository & Configure Environment
Set up the T-DEED workspace and disable interactive wandb prompts.

In [ ]:
import os

# Set working directory
WORKSPACE_DIR = "/kaggle/working"
TDEED_DIR = os.path.join(WORKSPACE_DIR, "T-DEED")

# Clone repository if not present
if not os.path.exists(TDEED_DIR):
    !git clone https://github.com/arturxe2/T-DEED.git {TDEED_DIR}

%cd {TDEED_DIR}

# Disable wandb interactive prompt
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_SILENT"] = "true"

--- 
## Step 4: Download SoccerNet Ball Action Spotting 2024 Dataset
We download videos to `/kaggle/temp/SoccerNet` to preserve Kaggle's `/kaggle/working` 20GB disk limit.

Executed in an isolated subprocess to ensure clean, conflict-free package loading.

In [ ]:
import os
import zipfile
import glob

DATA_DIR = "/kaggle/temp/SoccerNet"
os.makedirs(DATA_DIR, exist_ok=True)

print("📥 Downloading SoccerNet Ball Action Spotting 2024 annotations and videos (train & valid)...")
# Run in clean subprocess (completely avoids any Jupyter in-memory module caching)
!python -c "from SoccerNet.Downloader import SoccerNetDownloader as SNdl; d = SNdl(LocalDirectory='/kaggle/temp/SoccerNet'); d.downloadDataTask(task='spotting-ball-2024', split=['train', 'valid'])"

# Unzip all downloaded split files to extract game folders and labels
zip_files = glob.glob(os.path.join(DATA_DIR, "**", "*.zip"), recursive=True)
for zf in zip_files:
    print(f"📦 Unzipping {zf} to {DATA_DIR}...")
    with zipfile.ZipFile(zf, 'r') as ref:
        ref.extractall(DATA_DIR)
    try:
        os.remove(zf)  # Remove zip archive to save valuable Kaggle disk space
        print(f"Deleted archive {zf} to save disk space.")
    except Exception:
        pass

print("✅ Download & unzipping complete. Directory contents:")
!ls -lh {DATA_DIR}

--- 
## Step 5: Extract Frames at Model Input Resolution (796 × 448)
T-DEED processes extracted 25 fps JPEG frames scaled to $796 \times 448$.
We run the extraction script utilizing multi-processing workers into `/kaggle/temp/frames_snb`.
(Note: `extract_frames_snb.py` automatically appends `448` to the output folder name).

In [ ]:
import os

FRAMES_OUT = "/kaggle/temp/frames_snb"
os.makedirs(FRAMES_OUT, exist_ok=True)

print("🚀 Extracting frames from 720p videos...")
!python extract_frames_snb.py \
    --video_dir {DATA_DIR} \
    -o {FRAMES_OUT} \
    --sample_fps 25 \
    --num_workers 4

print("✅ Frames extraction finished.")

--- 
## Step 6: Configure T-DEED for Kaggle
We set up:
1. `data/soccernetball/labels_path.txt`: points to the unzipped SoccerNet data directory.
2. `config/SoccerNetBall/SoccerNetBall_kaggle.json`: specifies batch size, paths, learning rate, and backbone (`rny002_gsf`).

In [ ]:
import json
import os

# 1. Update label path pointer
LABELS_FILE = os.path.join(TDEED_DIR, "data", "soccernetball", "labels_path.txt")
os.makedirs(os.path.dirname(LABELS_FILE), exist_ok=True)
with open(LABELS_FILE, "w") as f:
    f.write(f"{DATA_DIR}\n")

print(f"✅ Updated labels path pointer: {DATA_DIR}")

# 2. Create optimized Kaggle configuration
SAVE_DIR = "/kaggle/working/checkpoints/SoccerNetBall_kaggle"
STORE_DIR = "/kaggle/temp/stored_partitions"
FRAMES_DIR = "/kaggle/temp/frames_snb448"  # Note: extract_frames_snb appends '448'
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(STORE_DIR, exist_ok=True)

kaggle_config = {
    "frame_dir": FRAMES_DIR,
    "save_dir": SAVE_DIR,
    "store_dir": STORE_DIR,
    "store_mode": "store",  # First run generates partitions, then switches to 'load'
    "batch_size": 4,
    "clip_len": 100,
    "crop_dim": -1,
    "dataset": "soccernetball",
    "radi_displacement": 4,
    "epoch_num_frames": 200000,
    "feature_arch": "rny002_gsf",  # RegNetY-200MF (lightweight, fits on T4)
    "learning_rate": 0.0008,
    "mixup": True,
    "modality": "rgb",
    "num_classes": 12,
    "num_epochs": 25,
    "warm_up_epochs": 3,
    "start_val_epoch": 5,
    "temporal_arch": "ed_sgp_mixer",
    "n_layers": 2,
    "sgp_ks": 9,
    "sgp_r": 4,
    "only_test": False,
    "criterion": "map",
    "num_workers": 2
}

CONFIG_PATH = os.path.join(TDEED_DIR, "config", "SoccerNetBall", "SoccerNetBall_kaggle.json")
with open(CONFIG_PATH, "w") as f:
    json.dump(kaggle_config, f, indent=4)

print(f"✅ Generated configuration: {CONFIG_PATH}")

--- 
## Step 7: Pre-Flight Check: Generate Partitions (`store_mode: store`)
T-DEED requires generating dataset clip index partitions on the first run. Once created, the script exits cleanly, after which we switch to `load` mode.

In [ ]:
# Run partition generation in store mode
!python train_tdeed.py --model SoccerNetBall_kaggle

# Switch config to 'load' mode for continuous training
with open(CONFIG_PATH, "r") as f:
    cfg = json.load(f)
cfg["store_mode"] = "load"
with open(CONFIG_PATH, "w") as f:
    json.dump(cfg, f, indent=4)

print("✅ Partitions cached successfully. Config switched to store_mode='load'.")

--- 
## Step 8: Train Model
Train T-DEED using the Linear Warmup + Cosine Annealing learning rate schedule.
Checkpoints are saved automatically to `/kaggle/working/checkpoints/SoccerNetBall_kaggle/`.

In [ ]:
!python train_tdeed.py --model SoccerNetBall_kaggle

--- 
## Step 9: Evaluate on Validation Split (mAP Calculation)
Run test evaluation across tight temporal tolerances (delta = 1s, 2s) to measure spotting precision for each of the 12 classes.

In [ ]:
# Set only_test = True in config
with open(CONFIG_PATH, "r") as f:
    cfg = json.load(f)
cfg["only_test"] = True
with open(CONFIG_PATH, "w") as f:
    json.dump(cfg, f, indent=4)

!python train_tdeed.py --model SoccerNetBall_kaggle

--- 
## Step 10: Run Inference Test on Video & Package Weights
Verify event spotting predictions on a video clip and package the best checkpoint for download.

In [ ]:
import glob
import shutil

# Locate best checkpoint
checkpoints = glob.glob(os.path.join(SAVE_DIR, "*.pt"))
print("Available checkpoints:", checkpoints)

# Package checkpoint and config into export directory
EXPORT_DIR = "/kaggle/working/tdeed_export"
os.makedirs(EXPORT_DIR, exist_ok=True)

for cp in checkpoints:
    shutil.copy(cp, EXPORT_DIR)
shutil.copy(CONFIG_PATH, EXPORT_DIR)

!tar -czvf /kaggle/working/tdeed_soccernetball_best.tar.gz -C {EXPORT_DIR} .
print("\n🎉 Model weights packaged to: /kaggle/working/tdeed_soccernetball_best.tar.gz")
print("You can now download this archive directly from Kaggle's Output panel!")